# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrabansal10/FlyRank_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [18]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/rudrabansal10/FlyRank_Internship/43b468d73eba109085f02d01f3a59754d5356453/data/raw/content_refresh_anonymized.csv")


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule

The rule identifies content that is most likely to benefit from a refresh.

Pages receive a higher score when they:
- are older,
- receive many search impressions,
- have a low click-through rate,
- have low engagement,
- have not been updated recently.

The score is intended as a decision-support ranking rather than a prediction.

The rule can produce one primary reason code.

- REFRESH_HIGH_IMPRESSIONS_LOW_CTR
  - The page receives substantial search impressions but has a relatively low click-through rate, suggesting that improving titles, metadata, or content quality may increase clicks.

Action Label:
- Refresh Content


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
# Higher score = higher refresh priority
def confidence(score):
    if score >= 0.90:
        return "High"
    elif score >= 0.75:
        return "Medium"
    else:
        return "Low"
def reason_code(row):
    if row["impressions_90d"] > 30000 and row["ctr"] < 0.5:
        return "HIGH_IMPRESSIONS_LOW_CTR"

    elif row["content_age_days"] > 365 and row["engagement_rate"] < 30:
        return "STALE_LOW_ENGAGEMENT"

    elif row["days_since_last_update"] > 180:
        return "STALE_CONTENT"

    else:
        return "GENERAL_REFRESH"

df["baseline_score"] = (
    0.35 * (df["impressions_90d"] / df["impressions_90d"].max()) +
    0.25 * (1 - df["ctr"] / df["ctr"].max()) +
    0.20 * (df["content_age_days"] / df["content_age_days"].max()) +
    0.20 * (1 - df["engagement_rate"] / 100)
)

df["reason_code"] = df.apply(reason_code, axis=1)

df["action"] = "Refresh Content"

df["confidence"] = df["baseline_score"].apply(confidence)

queue = df.sort_values("baseline_score", ascending=False)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Rank Reason_code	             Action	          Confidence


1    HIGH_IMPRESSIONS_LOW_CTR	 Refresh Content	High

2    HIGH_IMPRESSIONS_LOW_CTR	 Refresh Content	High

3    HIGH_IMPRESSIONS_LOW_CTR	 Refresh Content	High

4    HIGH_IMPRESSIONS_LOW_CTR	 Refresh Content	Medium

5    HIGH_IMPRESSIONS_LOW_CTR	 Refresh Content	Medium

6	   HIGH_IMPRESSIONS_LOW_CTR	 Refresh Content	Medium

7    HIGH_IMPRESSIONS_LOW_CTR	 Refresh Content	Medium

8	   HIGH_IMPRESSIONS_LOW_CTR	 Refresh Content	Medium

9    GENERAL_REFRESH	         Refresh Content	Medium

10   STALE_LOW_ENGAGEMENT	     Refresh Content	Medium

11   HIGH_IMPRESSIONS_LOW_CTR	 Refresh Content	Low

12   HIGH_IMPRESSIONS_LOW_CTR	 Refresh Content	Low

13   HIGH_IMPRESSIONS_LOW_CTR	 Refresh Content	Low

14   GENERAL_REFRESH	         Refresh Content	Low

15   HIGH_IMPRESSIONS_LOW_CTR	 Refresh Content	Low

16   HIGH_IMPRESSIONS_LOW_CTR	 Refresh Content	Low

17   STALE_LOW_ENGAGEMENT	     Refresh Content	Low

18   HIGH_IMPRESSIONS_LOW_CTR  Refresh Content	Low

19   HIGH_IMPRESSIONS_LOW_CTR	 Refresh Content	Low

20   HIGH_IMPRESSIONS_LOW_CTR	 Refresh Content	Low

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some recommendations may be incorrect because:

- Older pages are not always outdated.
- Low CTR can result from highly competitive search results rather than poor content.
- Pages with low engagement may intentionally answer simple questions quickly.
- Seasonal topics naturally fluctuate in performance.
- Some pages may have been recently refreshed, but the historical metrics have not yet reflected the changes.

Leakage check:

✔ trend_direction was not used.

✔ trend_pct was not used.

✔ is_declining_label was not used.

✔ No future-window metrics were used.

✔ Only historical content and performance metrics available at the decision time were used.

In [20]:
import json

metrics = {
    "top_20_size": 20,
    "rule_name": "baseline_action_score",
    "num_candidates": len(queue),
    "mean_score": float(queue["baseline_score"].mean()),
    "max_score": float(queue["baseline_score"].max())
}

with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)